<a href="https://colab.research.google.com/github/EllenEufrasio/Algoritmos_e_Logicas_de_Programacao/blob/main/notebook_exercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nome: Ellen Eufrasio de Oliveira
RA: 1131392413021

# Sessões 03 e 04 — Exercícios (entrega)

**Processamento de Linguagem Natural**

Para entrega de atividade. A célula abaixo reconstrói tudo que foi construído no `notebook_pratica.ipynb` (funções, dados, modelos) pra esse notebook rodar sozinho, sem precisar abrir o outro antes. Se você já rodou o notebook de prática nesta mesma sessão, pode rodar essa célula mesmo assim — não tem problema repetir.

In [3]:
# ===== Setup — reconstrói o que foi feito no notebook_pratica.ipynb =====
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from collections import Counter
import pandas as pd
import spacy

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)

try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    !pip install spacy
    !python -m spacy download pt_core_news_sm
    nlp = spacy.load("pt_core_news_sm")

stemmer = RSLPStemmer()
stopwords_pt = stopwords.words('portuguese')
stop_pt_set = set(stopwords_pt)
stop_pt_sem_negacao = stop_pt_set - {"não"}


# --- funções detectoras (seção 4) ---
def parece_plural(palavra):
    return palavra.lower().endswith('s')

def parece_diminutivo(palavra):
    return palavra.lower().endswith(('inho', 'inha', 'zinho', 'zinha'))

def parece_infinitivo(palavra):
    return palavra.lower().endswith(('ar', 'er', 'ir')) and len(palavra) > 3

def parece_voz_passiva(frase):
    padrao = r'\b(é|foi|são|foram|será|serão)\s+\w+(ado|ada|ados|adas|ido|ida|idos|idas)\b|\b\w+-se\b'
    return bool(re.search(padrao, frase.lower()))

def contem_linguagem_inadequada(frase, lista_proibida):
    tokens = frase.lower().split()
    return any(t.strip('.,!?') in lista_proibida for t in tokens)

lista_proibida = {"idiota", "burro", "imbecil"}


# --- pipeline (seções 8 e 13) ---
def limpar_texto(texto, remover_stopwords=True):
    tokens = word_tokenize(texto.lower(), language='portuguese')
    tokens = [t for t in tokens if t.isalpha()]
    if remover_stopwords:
        tokens = [t for t in tokens if t not in stopwords_pt]
    return tokens


def limpar_texto_v2(texto, remover_stopwords=True, reduzir=None, stopset=None):
    doc = nlp(texto.lower())
    sw = stopset if stopset is not None else stop_pt_set
    tokens = []
    for tok in doc:
        if not tok.is_alpha:
            continue
        if remover_stopwords and tok.text in sw:
            continue
        forma = tok.lemma_ if reduzir == 'lema' else tok.text
        tokens.append(forma)
    if reduzir == 'stem':
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens


# --- dados usados nos exercícios (seções 1, 5 e 7) ---
frase_dificil = "Vi no site https://loja.com que o guarda-chuva tá R$ 29,90 — não é caro pelo que é."

texto_avaliacao = (
    "Comprei esse fone de ouvido semana passada e, sinceramente, não esperava tanto. "
    "A bateria dura o dia inteiro, o som é ótimo pra esse preço e o app conectou super rápido. "
    "Só achei a caixinha um pouco frágil, mas isso não estragou minha experiência. Recomendo!"
)

url_hatebr = "https://raw.githubusercontent.com/franciellevargas/HateBR/main/dataset/HateBR.csv"
hatebr = pd.read_csv(url_hatebr)
comentarios_hatebr_leve = hatebr[hatebr["id"].isin([1, 2, 3, 8, 6999])]

print("Setup pronto.")

Setup pronto.


Sua vez


**1.** Pegue o nome de arquivo `"Relatório Final - Março 2026.pdf"` e escreva um pré-processamento (regex + o que mais precisar) que deixe o nome no formato que um bucket S3 da AWS aceita sem erro: sem espaço, sem acento, sem cedilha, tudo minúsculo, só letras/números/ponto/underscore. Resultado esperado: `"relatorio_final_marco_2026.pdf"`.

In [20]:
import unicodedata

nome_arquivo = ["Relatório Final - Março 2026.pdf", "Relação da Educação com a Política", ]

# Dica: unicodedata.normalize('NFKD', texto) separa a letra do acento;
# .encode('ascii', 'ignore').decode('ascii') descarta o que sobrou (acento, cedilha)
# Depois é regex: tudo minúsculo, espaço/pontuação vira "_", só sobra letra/número/ponto/underscore

# nome_limpo = ...
# print(nome_limpo)

nome_arquivo = "Relatório Final - Março 2026.pdf"
nome_limpo = unicodedata.normalize('NFKD', nome_arquivo)
nome_limpo = nome_limpo.encode('ascii', 'ignore').decode('ascii')
nome_limpo = nome_limpo.lower()
nome_limpo = re.sub(r'[^a-z0-9._]+', '_', nome_limpo)
nome_limpo = nome_limpo.strip('_')

print(nome_limpo)

relatorio_final_marco_2026.pdf


**2.** Pegue o texto `"Adorei o show da @banda123 ontem! #show #musica"` e escreva regex que remova menções (`@usuario`) e hashtags (`#assunto`) — comum antes de indexar post de rede social, já que essas marcações não ajudam a entender o assunto do texto. Resultado esperado (sem espaço duplo sobrando): `"Adorei o show da ontem!"`.

In [29]:
post = "Adorei o show da @banda123 ontem! #show #musica"

post_limpo = re.sub(r'\s+', ' ', post)
post_limpo = re.sub(r'@\w+', '', post_limpo)
post_limpo = re.sub(r'#\w+', '', post_limpo)
#post_limpo = post_limpo.strip()
post_limpo = re.sub(r'\s+', ' ', post_limpo)
print(post_limpo)

Adorei o show da ontem! 


**3.** Pegue os três números `"(11) 98765-4321"`, `"11 98765 4321"` e `"11987654321"` e escreva regex que normalize os três pro **mesmo formato**, só dígitos — útil pra comparar/deduplicar telefones que chegaram digitados de jeitos diferentes num cadastro. Resultado esperado nos três casos: `"11987654321"`.

In [33]:
telefones = ["(11) 98765-4321", "11 98765 4321", "11987654321"]

for tel in telefones:
    tel_normalizado = re.sub(r'\D', '', tel)
    print("O números normalizado é: ", tel_normalizado)


O números normalizado é:  11987654321
O números normalizado é:  11987654321
O números normalizado é:  11987654321


**4.** Crie uma função `detecta_pessoalidade(texto)` que identifica se um texto viola a impessoalidade acadêmica (marcas de 1ª pessoa) — devolve `True` se achar violação, `False` se o texto está impessoal.

Requisitos:
- Case-insensitive.
- Detectar pronomes **isolados** (palavra inteira, não substring): `eu, nós, meu, minha, meus, minhas, nosso, nossa, nossos, nossas, me, nos`. Cuidado pra não confundir com palavra que só *contém* essas letras no meio (ex.: "pneu" não deve acionar o "eu").
- Detectar verbo na 1ª pessoa do plural pelos sufixos `-amos`, `-emos`, `-imos`, `-ímos` (repare o acento — "concluímos" não é o mesmo sufixo de "partimos") e `-íamos`.
- Cuidado com substantivo que termina parecido com verbo (ex.: "os termos do contrato", "os primos chegaram") — a regra de sufixo sozinha erra nesses casos; pense em como contornar (dica: onde mais nesse notebook você já resolveu um problema parecido?).

Use `.split()` (ou regex) combinado com `.endswith()` pra checar o final de cada palavra.

In [34]:
#Rascunho

import re

def detecta_pessoalidade(texto):
    # SUA LÓGICA AQUI
    pass


# --- Casos de teste ---
frases_teste = [
    ("O projeto foi concluído com sucesso.", False),
    ("Nosso projeto foi concluído com sucesso.", True),                        # Pronome "Nosso"
    ("Eu analisei os dados da pesquisa.", True),                               # Pronome "Eu"
    ("A análise dos dados da pesquisa evidenciou falhas.", False),
    ("Nesta etapa, iremos realizar a coleta de amostras.", True),              # Verbo "iremos"
    ("Fizemos os testes iniciais no laboratório.", True),                     # Verbo "Fizemos"
    ("Se houvesse mais tempo, faríamos novos ensaios.", True),                # Verbo "faríamos"
    ("Os termos do contrato foram revisados.", False),                       # Falso positivo clássico p/ "-mos"
    ("O pneu do carro furou durante o trajeto.", False),                     # Falso positivo clássico p/ "eu"
    ("Concluímos que a hipótese era verdadeira.", True),                     # Verbo "Concluímos" — repare o acento no í
    ("Os primos chegaram cedo para o almoço em família.", False),           # Outro falso positivo — "primos" não é verbo
]

for frase, resultado_esperado in frases_teste:
    resultado_obtido = detecta_pessoalidade(frase)
    status = "PASSOU" if resultado_obtido == resultado_esperado else "FALHOU"
    print(f"{status} | Frase: {frase!r}")

FALHOU | Frase: 'O projeto foi concluído com sucesso.'
FALHOU | Frase: 'Nosso projeto foi concluído com sucesso.'
FALHOU | Frase: 'Eu analisei os dados da pesquisa.'
FALHOU | Frase: 'A análise dos dados da pesquisa evidenciou falhas.'
FALHOU | Frase: 'Nesta etapa, iremos realizar a coleta de amostras.'
FALHOU | Frase: 'Fizemos os testes iniciais no laboratório.'
FALHOU | Frase: 'Se houvesse mais tempo, faríamos novos ensaios.'
FALHOU | Frase: 'Os termos do contrato foram revisados.'
FALHOU | Frase: 'O pneu do carro furou durante o trajeto.'
FALHOU | Frase: 'Concluímos que a hipótese era verdadeira.'
FALHOU | Frase: 'Os primos chegaram cedo para o almoço em família.'


In [55]:
def detecta_pessoalidade(texto):
    pronomes = {"eu","nós","meu","minha","meus","minhas",
                "nosso","nossa","nossos","nossas","me","nos"}
    tokens = re.findall(r"\b\w+\b", texto.lower())

    for token in tokens:
        if token in pronomes:
            return True

    sufixos = ["amos","emos","imos","ímos","íamos"]

    #para evitar problemas com a pala vvra "primo" como ciatdo no enunciado
    excecoes = {"termos","primos", "primo"}
    for token in tokens:
        if len(token) > 4 and any(token.endswith(suf) for suf in sufixos):
            if token not in excecoes:
                return True

    return False


# --- Casos de teste ---
frases_teste = [
    ("O projeto foi concluído com sucesso.", False),
    ("Nosso projeto foi concluído com sucesso.", True),                        # Pronome "Nosso"
    ("Eu analisei os dados da pesquisa.", True),                               # Pronome "Eu"
    ("A análise dos dados da pesquisa evidenciou falhas.", False),
    ("Nesta etapa, iremos realizar a coleta de amostras.", True),              # Verbo "iremos"
    ("Fizemos os testes iniciais no laboratório.", True),                     # Verbo "Fizemos"
    ("Se houvesse mais tempo, faríamos novos ensaios.", True),                # Verbo "faríamos"
    ("Os termos do contrato foram revisados.", False),                       # Falso positivo clássico p/ "-mos"
    ("O pneu do carro furou durante o trajeto.", False),                     # Falso positivo clássico p/ "eu"
    ("Concluímos que a hipótese era verdadeira.", True),                     # Verbo "Concluímos" — repare o acento no í
    ("Os primos chegaram cedo para o almoço em família.", False),           # Outro falso positivo — "primos" não é verbo
]

for frase, resultado_esperado in frases_teste:
    resultado_obtido = detecta_pessoalidade(frase)
    status = "PASSOU" if resultado_obtido == resultado_esperado else "FALHOU"
    print(f"{status} | Frase: {frase!r}")


PASSOU | Frase: 'O projeto foi concluído com sucesso.'
PASSOU | Frase: 'Nosso projeto foi concluído com sucesso.'
PASSOU | Frase: 'Eu analisei os dados da pesquisa.'
PASSOU | Frase: 'A análise dos dados da pesquisa evidenciou falhas.'
PASSOU | Frase: 'Nesta etapa, iremos realizar a coleta de amostras.'
PASSOU | Frase: 'Fizemos os testes iniciais no laboratório.'
PASSOU | Frase: 'Se houvesse mais tempo, faríamos novos ensaios.'
PASSOU | Frase: 'Os termos do contrato foram revisados.'
PASSOU | Frase: 'O pneu do carro furou durante o trajeto.'
PASSOU | Frase: 'Concluímos que a hipótese era verdadeira.'
PASSOU | Frase: 'Os primos chegaram cedo para o almoço em família.'


### 5. Contexto

Você foi contratado(a) como analista de dados por uma agência de Marketing Digital. Uma das principais estratégias para fazer um texto aparecer na primeira página do Google (uma técnica chamada SEO) é garantir que a "palavra-chave" principal do artigo seja repetida um número ideal de vezes.

Seu primeiro desafio é criar uma ferramenta automatizada que leia o texto de um blog e conte exatamente quantas vezes uma palavra específica aparece nele.

### A Tarefa

Com base nos seus conhecimentos de manipulação de strings e Processamento de Linguagem Natural (PLN) básico, crie uma função em Python chamada `conta_palavra(texto, palavra_alvo)`.

A função deve receber um texto completo e a palavra que queremos buscar, e deve retornar um número inteiro representando a quantidade de vezes que aquela palavra exata aparece no texto.

### Requisitos e Pegadinhas (Atenção!)

1. **Ignorar Maiúsculas/Minúsculas:** Se a palavra alvo for `"python"`, a função deve contar `"Python"`, `"PYTHON"` ou `"pyThon"` como ocorrências válidas.
2. **Cuidado com a Pontuação:** As palavras em um texto costumam vir grudadas com sinais de pontuação. `"python."` ou `"python,"` devem ser contabilizadas corretamente.
3. **Palavra Exata (Falsos Positivos):** Se a palavra alvo for `"sol"`, a função **NÃO** deve contar palavras que apenas contenham essas letras, como `"girassol"`, `"solidão"` ou `"solo"`. Apenas a palavra `"sol"` isolada conta!

### Dicas de Implementação

* Você pode resolver isso usando **Expressões Regulares** (`re.findall()` usando `\b` para limites de palavra).
* **OU** você pode resolver usando manipulação de strings: transformar tudo em minúsculo (`.lower()`), quebrar o texto em uma lista de palavras (`.split()`) e limpar a pontuação das pontas de cada palavra (usando `.strip('.,!?')`).

In [56]:
#RASCUNHOO
# estrutura base

import re

def conta_palavra(texto, palavra_alvo):
    # SUA LÓGICA AQUI
    pass


# --- Casos de Teste pode colocar váios outros ---
texto_exemplo = """
O sol brilhava forte no céu. Girassóis adoram o sol, mas a solidão do deserto
castigava. Sob o Sol quente, o solo rachava. Que sol maravilhoso!
"""

testes = [
    ("sol", 4),       # 'sol', 'sol,', 'Sol', 'sol' (ignorar girassol, solidão e solo)
    ("girassóis", 1), # Tem que lidar com maiúscula e pontuação
    ("chuva", 0),     # Palavra que não existe no texto
    ("o", 4)          # O (maiúsculo no início), o, o, o
]

# Código para testar sua função:
for palavra, resultado_esperado in testes:
    resultado_obtido = conta_palavra(texto_exemplo, palavra)
    status = "✅ PASSOU" if resultado_obtido == resultado_esperado else "❌ FALHOU"
    print(f"{status} | Alvo: '{palavra}' | Esperado: {resultado_esperado} | Obtido: {resultado_obtido}")

❌ FALHOU | Alvo: 'sol' | Esperado: 4 | Obtido: None
❌ FALHOU | Alvo: 'girassóis' | Esperado: 1 | Obtido: None
❌ FALHOU | Alvo: 'chuva' | Esperado: 0 | Obtido: None
❌ FALHOU | Alvo: 'o' | Esperado: 4 | Obtido: None


In [57]:
import re

def conta_palavra(texto, palavra_alvo):
    # Normaliza tudo para minúsculas
    texto = texto.lower()
    palavra_alvo = palavra_alvo.lower()

    # Regex com \b para garantir palavra isolada
    # \b = limite de palavra (evita contar "girassol" quando alvo é "sol")
    padrao = r'\b' + re.escape(palavra_alvo) + r'\b'

    # Busca todas as ocorrências, ignorando pontuação
    ocorrencias = re.findall(padrao, texto, flags=re.IGNORECASE)

    return len(ocorrencias)


# --- Casos de Teste ---
texto_exemplo = """
O sol brilhava forte no céu. Girassóis adoram o sol, mas a solidão do deserto
castigava. Sob o Sol quente, o solo rachava. Que sol maravilhoso!
"""

testes = [
    ("sol", 4),       # 'sol', 'sol,', 'Sol', 'sol' (ignorar girassol, solidão e solo)
    ("girassóis", 1), # Tem que lidar com maiúscula e pontuação
    ("chuva", 0),     # Palavra que não existe no texto
    ("o", 4)          # O (maiúsculo no início), o, o, o
]

# Código para testar sua função:
for palavra, resultado_esperado in testes:
    resultado_obtido = conta_palavra(texto_exemplo, palavra)
    status = "✅ PASSOU" if resultado_obtido == resultado_esperado else "❌ FALHOU"
    print(f"{status} | Alvo: '{palavra}' | Esperado: {resultado_esperado} | Obtido: {resultado_obtido}")


✅ PASSOU | Alvo: 'sol' | Esperado: 4 | Obtido: 4
✅ PASSOU | Alvo: 'girassóis' | Esperado: 1 | Obtido: 1
✅ PASSOU | Alvo: 'chuva' | Esperado: 0 | Obtido: 0
✅ PASSOU | Alvo: 'o' | Esperado: 4 | Obtido: 4


6. Crie um função usando Regex ou funçõe nativas em python que após uma tokenização, você consiga extrair de uma frase apenas um valor monetário,

Ex: O Sabão custou R$ 20,00;  

O Sabão custou R$ 20,00 ou R$ 20.00;

In [50]:
import re

def extrair_valor(frase):
    padrao = r'R\$ ?\d+(?:[.,]\d{2})?'
    resultado = re.findall(padrao, frase)
    return resultado

frase1 = "O Sabão custou R$ 20,00"

print(extrair_valor(frase1))

['R$ 20,00']
